## MAST vs edgeR

Let's examine the overlap and unique DEGs found through MAST and edgeR

In [3]:
import pandas as pd
import os

data_dir = '/Users/jack/seq/analysis/combined/0_all-sample/DGE_filtered'

mast_SEv1h = pd.read_csv(os.path.join(data_dir, 'DEG_MAST_SE-v-1h_combined.csv'))
mast_SEv6h = pd.read_csv(os.path.join(data_dir, 'DEG_MAST_SE-v-6h_combined.csv'))
mast_1hv6h = pd.read_csv(os.path.join(data_dir, 'DEG_MAST_1h-v-6h_combined.csv'))

edgeR_SEv1h = pd.read_csv(os.path.join(data_dir, 'DEG_edgeR_SE-v-1h_combined.csv'))
edgeR_SEv6h = pd.read_csv(os.path.join(data_dir, 'DEG_edgeR_SE-v-6h_combined.csv'))
edgeR_1hv6h = pd.read_csv(os.path.join(data_dir, 'DEG_edgeR_1h-v-6h_combined.csv'))

### SE v 1h

In [8]:
# number of edgeR DEG (FDR < 0.05)
print('edgeR DEG (FDR < 0.05)')
print('SE-v-1h:', edgeR_SEv1h.shape[0])
print('SE-v-6h:', edgeR_SEv6h[edgeR_SEv6h['FDR'] < 0.05].shape[0])
print('1h-v-6h:', edgeR_1hv6h[edgeR_1hv6h['FDR'] < 0.05].shape[0])

edgeR DEG (FDR < 0.05)
SE-v-1h: 13
SE-v-6h: 0
1h-v-6h: 0


In [4]:
# sort out only FDR < 0.05 from edgeR results, since that returns all genes
edgeR_SEv1h = edgeR_SEv1h[edgeR_SEv1h['FDR'] < 0.05]
edgeR_SEv1h

,Unnamed: 0,logFC,logCPM,F,PValue,FDR,gene
0,Mir670hg,1.362609,4.632315,76.167647,6.972020e-07,0.005896,Mir670hg
1,Sik2,1.028570,7.181167,74.878270,7.689708e-07,0.005896,Sik2
2,Tnfrsf25,1.542985,4.114458,62.032063,2.228288e-06,0.007827,Tnfrsf25
3,Ntrk2,0.866295,9.945665,60.947252,2.458047e-06,0.007827,Ntrk2
4,Kdm7a,1.001227,6.836766,60.537788,2.551769e-06,0.007827,Kdm7a
5,Osbpl6,0.647359,9.283349,53.558967,4.998716e-06,0.012777,Osbpl6
6,Sik3,0.660254,8.630848,47.528534,9.492222e-06,0.019792,Sik3
7,Gm29865,-1.315066,3.776909,46.780380,1.032453e-05,0.019792,Gm29865
8,Map2,0.586760,9.414053,41.931411,1.829533e-05,0.031175,Map2
9,Gfra1,1.114583,5.431543,38.442672,2.853266e-05,0.039831,Gfra1


In [5]:
# shared DEGs found by mast and edgeR
shared_genes_SEv1h = set(mast_SEv1h['primerid'].to_list()).intersection(set(edgeR_SEv1h[edgeR_SEv1h['FDR'] < 0.05]['gene'].to_list()))
shared_genes_SEv1h

{'Gfra1',
 'Gm29865',
 'Kdm7a',
 'Map2',
 'Mir670hg',
 'Oga',
 'Osbpl6',
 'Sik2',
 'Sik3',
 'Specc1',
 'Tnfrsf25',
 'Zfp46'}

In [9]:
# genes in edgeR_SEv1h not in shared_genes_SEv1h
edgeR_SEv1h[~edgeR_SEv1h['gene'].isin(shared_genes_SEv1h)]

,Unnamed: 0,logFC,logCPM,F,PValue,FDR,gene
3,Ntrk2,0.866295,9.945665,60.947252,0.000002,0.007827,Ntrk2


In [10]:
# make a df of shared genes
shared_genes_SEv1h_df = pd.DataFrame(shared_genes_SEv1h, columns=['gene'])

# merge in results from MAST
shared_genes_SEv1h_df = pd.merge(shared_genes_SEv1h_df, mast_SEv1h, left_on='gene', right_on='primerid', how='left')
shared_genes_SEv1h_df.rename(columns={'FDR': 'FDR_mast'}, inplace=True)

# merge in results from edgeR
shared_genes_SEv1h_df = pd.merge(shared_genes_SEv1h_df, edgeR_SEv1h, left_on='gene', right_on='gene', how='left')
shared_genes_SEv1h_df.rename(columns={'FDR': 'FDR_edgeR'}, inplace=True)

shared_genes_SEv1h_df

,gene,primerid,Pr(>Chisq),coef,FDR_mast,Unnamed: 0,logFC,logCPM,F,PValue,FDR_edgeR
0,Sik3,Sik3,5.603759e-07,0.457399,0.000388,Sik3,0.660254,8.630848,47.528534,9.492222e-06,0.019792
1,Oga,Oga,1.074322e-07,0.292833,0.000167,Oga,0.738153,6.089920,36.978511,3.470200e-05,0.044349
2,Tnfrsf25,Tnfrsf25,1.390247e-05,0.195232,0.002368,Tnfrsf25,1.542985,4.114458,62.032063,2.228288e-06,0.007827
3,Sik2,Sik2,1.231024e-08,0.440675,0.000068,Sik2,1.028570,7.181167,74.878270,7.689708e-07,0.005896
4,Gfra1,Gfra1,9.777747e-05,0.158103,0.008200,Gfra1,1.114583,5.431543,38.442672,2.853266e-05,0.039831
5,Osbpl6,Osbpl6,2.910835e-06,0.469172,0.001007,Osbpl6,0.647359,9.283349,53.558967,4.998716e-06,0.012777
6,Map2,Map2,3.585553e-08,0.519746,0.000099,Map2,0.586760,9.414053,41.931411,1.829533e-05,0.031175
7,Gm29865,Gm29865,6.161477e-07,-0.108896,0.000401,Gm29865,-1.315066,3.776909,46.780380,1.032453e-05,0.019792
8,Specc1,Specc1,2.819808e-07,0.460743,0.000240,Specc1,0.622599,8.597023,36.387467,3.761794e-05,0.044378
9,Zfp46,Zfp46,4.543269e-06,-0.104148,0.001397,Zfp46,-1.219588,3.795909,38.432906,2.856940e-05,0.039831
